# High-Dimensional Data Clustering Framework on Cloud with Deep AI

This notebook demonstrates a comprehensive approach to clustering high-dimensional data using deep learning techniques deployed on cloud infrastructure.

**Authors**: Research Team  
**Date**: August 2025  
**Framework**: TensorFlow/PyTorch + Azure Cloud Services  

## Table of Contents
1. [Environment Setup and Library Installation](#1)
2. [Cloud Storage Configuration](#2)
3. [Data Loading and Preprocessing](#3)
4. [Dimensionality Reduction with Deep Learning](#4)
5. [Neural Network-Based Feature Extraction](#5)
6. [Deep Clustering Model Implementation](#6)
7. [Cloud-Based Model Training](#7)
8. [Cluster Evaluation and Visualization](#8)
9. [Scalable Inference Pipeline](#9)
10. [Model Deployment and Monitoring](#10)

<a id='1'></a>
## 1. Environment Setup and Library Installation

First, let's install and import all required libraries for our clustering framework.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install torch torchvision tensorflow azure-storage-blob azure-identity
# !pip install scikit-learn umap-learn hdbscan plotly seaborn
# !pip install mlflow wandb optuna

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Deep learning frameworks
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Scikit-learn for traditional ML
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, SpectralClustering
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

# Advanced clustering
import umap.umap_ as umap
import hdbscan

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

# Azure SDK
from azure.storage.blob import BlobServiceClient
from azure.identity import DefaultAzureCredential

# MLOps and Experiment Tracking
import mlflow
import mlflow.pytorch
import wandb

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
tf.random.set_seed(42)

# Configure plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print(f"✅ Environment setup complete!")
print(f"PyTorch version: {torch.__version__}")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

<a id='2'></a>
## 2. Cloud Storage Configuration

Configure Azure Blob Storage for scalable data storage and retrieval.

In [ ]:
# Azure Storage Configuration
class AzureStorageManager:
    def __init__(self, account_name: str, container_name: str = "clustering-data"):
        self.account_name = account_name
        self.container_name = container_name
        
        # Use DefaultAzureCredential for authentication
        credential = DefaultAzureCredential()
        account_url = f"https://{account_name}.blob.core.windows.net"
        
        self.blob_service_client = BlobServiceClient(
            account_url=account_url, 
            credential=credential
        )
    
    def upload_data(self, local_path: Path, blob_name: str):
        """Upload local file to Azure Blob Storage"""
        blob_client = self.blob_service_client.get_blob_client(
            container=self.container_name, blob=blob_name
        )
        
        with open(local_path, "rb") as data:
            blob_client.upload_blob(data, overwrite=True)
        
        print(f"✅ Uploaded {local_path} to {blob_name}")
    
    def download_data(self, blob_name: str, local_path: Path):
        """Download file from Azure Blob Storage"""
        blob_client = self.blob_service_client.get_blob_client(
            container=self.container_name, blob=blob_name
        )
        
        with open(local_path, "wb") as download_file:
            download_file.write(blob_client.download_blob().readall())
        
        print(f"✅ Downloaded {blob_name} to {local_path}")
    
    def list_blobs(self, prefix: str = ""):
        """List all blobs in the container"""
        container_client = self.blob_service_client.get_container_client(
            self.container_name
        )
        
        blobs = []
        for blob in container_client.list_blobs(name_starts_with=prefix):
            blobs.append(blob.name)
        
        return blobs

# Configuration (replace with your Azure Storage account)
# storage_manager = AzureStorageManager("your-storage-account")
print("💾 Azure Storage Manager configured")

<a id='3'></a>
## 3. Data Loading and Preprocessing

Load and preprocess high-dimensional datasets with comprehensive data cleaning.

In [ ]:
class DataPreprocessor:
    def __init__(self):
        self.scaler = StandardScaler()
        self.feature_names = None
        self.original_shape = None
    
    def load_data(self, file_path: Path):
        """Load data from various file formats"""
        if file_path.suffix == '.csv':
            df = pd.read_csv(file_path)
        elif file_path.suffix == '.parquet':
            df = pd.read_parquet(file_path)
        elif file_path.suffix == '.json':
            df = pd.read_json(file_path)
        else:
            raise ValueError(f"Unsupported file format: {file_path.suffix}")
        
        print(f"📊 Loaded data: {df.shape}")
        return df
    
    def analyze_data_quality(self, df: pd.DataFrame):
        """Comprehensive data quality analysis"""
        print("🔍 Data Quality Analysis")
        print(f"Shape: {df.shape}")
        print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        
        # Missing values
        missing = df.isnull().sum()
        missing_pct = (missing / len(df)) * 100
        missing_info = pd.DataFrame({
            'Missing Count': missing,
            'Missing Percentage': missing_pct
        }).sort_values('Missing Count', ascending=False)
        
        if missing.sum() > 0:
            print(f"\n⚠️  Missing values found:")
            print(missing_info[missing_info['Missing Count'] > 0])
        else:
            print("✅ No missing values")
        
        # Data types
        print(f"\n📈 Data types:")
        print(df.dtypes.value_counts())
        
        # Numerical columns statistics
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        if len(numeric_cols) > 0:
            print(f"\n📊 Numerical features: {len(numeric_cols)}")
            print(df[numeric_cols].describe())
        
        return missing_info
    
    def handle_missing_values(self, df: pd.DataFrame, strategy: str = 'mean'):
        """Handle missing values with various strategies"""
        if df.isnull().sum().sum() == 0:
            return df
        
        print(f"🛠️  Handling missing values with strategy: {strategy}")
        
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        categorical_cols = df.select_dtypes(include=['object', 'category']).columns
        
        df_clean = df.copy()
        
        # Handle numeric columns
        if strategy == 'mean':
            df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].mean())
        elif strategy == 'median':
            df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].median())
        elif strategy == 'mode':
            for col in numeric_cols:
                df_clean[col] = df_clean[col].fillna(df_clean[col].mode().iloc[0] if not df_clean[col].mode().empty else 0)
        
        # Handle categorical columns
        for col in categorical_cols:
            df_clean[col] = df_clean[col].fillna(df_clean[col].mode().iloc[0] if not df_clean[col].mode().empty else 'Unknown')
        
        print(f"✅ Missing values handled")
        return df_clean
    
    def remove_outliers(self, df: pd.DataFrame, method: str = 'iqr', threshold: float = 1.5):
        """Remove outliers using IQR or Z-score method"""
        numeric_cols = df.select_dtypes(include=[np.number]).columns
        df_clean = df.copy()
        
        if method == 'iqr':
            for col in numeric_cols:
                Q1 = df_clean[col].quantile(0.25)
                Q3 = df_clean[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - threshold * IQR
                upper_bound = Q3 + threshold * IQR
                df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
        
        elif method == 'zscore':
            from scipy import stats
            for col in numeric_cols:
                z_scores = np.abs(stats.zscore(df_clean[col]))
                df_clean = df_clean[z_scores < threshold]
        
        removed_samples = len(df) - len(df_clean)
        print(f"🚮 Removed {removed_samples} outliers ({removed_samples/len(df)*100:.2f}%)")
        
        return df_clean
    
    def prepare_features(self, df: pd.DataFrame, target_col: str = None):
        """Prepare features for clustering"""
        # Separate features and target
        if target_col and target_col in df.columns:
            y = df[target_col].values
            X_df = df.drop(columns=[target_col])
        else:
            y = None
            X_df = df.copy()
        
        # Select only numeric columns
        numeric_cols = X_df.select_dtypes(include=[np.number]).columns
        X_df = X_df[numeric_cols]
        
        self.feature_names = list(X_df.columns)
        X = X_df.values
        self.original_shape = X.shape
        
        print(f"✅ Prepared {X.shape[0]} samples with {X.shape[1]} features")
        
        return X, y, self.feature_names
    
    def normalize_data(self, X: np.ndarray, method: str = 'standard'):
        """Normalize data using different methods"""
        if method == 'standard':
            X_norm = self.scaler.fit_transform(X)
        elif method == 'minmax':
            scaler = MinMaxScaler()
            X_norm = scaler.fit_transform(X)
        elif method == 'robust':
            from sklearn.preprocessing import RobustScaler
            scaler = RobustScaler()
            X_norm = scaler.fit_transform(X)
        else:
            X_norm = X
        
        print(f"✅ Data normalized using {method} method")
        return X_norm

# Initialize preprocessor
preprocessor = DataPreprocessor()
print("🔧 Data preprocessor initialized")

In [ ]:
# Generate sample high-dimensional dataset for demonstration
from sklearn.datasets import make_blobs, make_classification

def generate_sample_dataset(n_samples=2000, n_features=500, n_clusters=5):
    """Generate sample high-dimensional dataset"""
    print(f"🎲 Generating sample dataset: {n_samples} samples, {n_features} features, {n_clusters} clusters")
    
    # Create base clusters
    X_base, y_true = make_blobs(
        n_samples=n_samples,
        centers=n_clusters,
        n_features=min(50, n_features),
        random_state=42,
        cluster_std=1.5
    )
    
    # Add noise features
    if n_features > 50:
        noise_features = np.random.randn(n_samples, n_features - 50) * 0.5
        X = np.column_stack([X_base, noise_features])
    else:
        X = X_base
    
    # Create feature names
    feature_names = [f"feature_{i:03d}" for i in range(X.shape[1])]
    
    # Create DataFrame
    df = pd.DataFrame(X, columns=feature_names)
    df['true_label'] = y_true
    
    # Add some missing values randomly
    missing_mask = np.random.random(X.shape) < 0.02  # 2% missing values
    df_with_missing = df.copy()
    df_with_missing.iloc[:, :-1] = df_with_missing.iloc[:, :-1].mask(missing_mask[:, :-1] if missing_mask.shape[1] > 1 else missing_mask)
    
    print(f"✅ Generated dataset with shape: {df_with_missing.shape}")
    return df_with_missing, y_true

# Generate sample data
sample_df, true_labels = generate_sample_dataset(n_samples=2000, n_features=200, n_clusters=5)

# Analyze data quality
missing_info = preprocessor.analyze_data_quality(sample_df)

# Handle missing values
clean_df = preprocessor.handle_missing_values(sample_df, strategy='mean')

# Prepare features
X, y, feature_names = preprocessor.prepare_features(clean_df, target_col='true_label')

# Normalize data
X_normalized = preprocessor.normalize_data(X, method='standard')

print(f"\n🎯 Final dataset ready for clustering:")
print(f"   - Samples: {X_normalized.shape[0]}")
print(f"   - Features: {X_normalized.shape[1]}")
print(f"   - True clusters: {len(np.unique(y))}")

<a id='4'></a>
## 4. Dimensionality Reduction with Deep Learning

Implement autoencoders and variational autoencoders for dimensionality reduction.

In [ ]:
class DeepAutoEncoder(nn.Module):
    """Deep Autoencoder for dimensionality reduction"""
    
    def __init__(self, input_dim, latent_dim=32, hidden_dims=[512, 256, 128]):
        super(DeepAutoEncoder, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        self.hidden_dims = hidden_dims
        
        # Encoder
        encoder_layers = []
        in_dim = input_dim
        
        for hidden_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            in_dim = hidden_dim
        
        encoder_layers.append(nn.Linear(in_dim, latent_dim))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Decoder
        decoder_layers = []
        in_dim = latent_dim
        
        for hidden_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            ])
            in_dim = hidden_dim
        
        decoder_layers.append(nn.Linear(in_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z


class VariationalAutoEncoder(nn.Module):
    """Variational Autoencoder for probabilistic dimensionality reduction"""
    
    def __init__(self, input_dim, latent_dim=32, hidden_dims=[512, 256]):
        super(VariationalAutoEncoder, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # Encoder
        encoder_layers = []
        in_dim = input_dim
        
        for hidden_dim in hidden_dims:
            encoder_layers.extend([
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3)
            ])
            in_dim = hidden_dim
        
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Latent space
        self.fc_mu = nn.Linear(hidden_dims[-1], latent_dim)
        self.fc_logvar = nn.Linear(hidden_dims[-1], latent_dim)
        
        # Decoder
        decoder_layers = []
        in_dim = latent_dim
        
        for hidden_dim in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3)
            ])
            in_dim = hidden_dim
        
        decoder_layers.append(nn.Linear(in_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
    
    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar, z
    
    def loss_function(self, x_recon, x, mu, logvar):
        """VAE loss function"""
        recon_loss = F.mse_loss(x_recon, x, reduction='sum')
        kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        return recon_loss + kld_loss, recon_loss, kld_loss


def train_autoencoder(model, train_loader, epochs=100, lr=0.001, device='cpu'):
    """Train autoencoder model"""
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
    
    model.train()
    losses = []
    
    print(f"🚀 Training {model.__class__.__name__} for {epochs} epochs")
    
    for epoch in range(epochs):
        epoch_loss = 0
        
        for batch_idx, batch in enumerate(train_loader):
            batch = batch[0].to(device)  # Get data tensor
            optimizer.zero_grad()
            
            if isinstance(model, VariationalAutoEncoder):
                x_recon, mu, logvar, z = model(batch)
                loss, recon_loss, kld_loss = model.loss_function(x_recon, batch, mu, logvar)
            else:
                x_recon, z = model(batch)
                loss = F.mse_loss(x_recon, batch)
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        scheduler.step()
        avg_loss = epoch_loss / len(train_loader)
        losses.append(avg_loss)
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d}/{epochs}: Loss = {avg_loss:.6f}")
    
    print(f"✅ Training completed!")
    return losses


# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Using device: {device}")

# Prepare data for PyTorch
X_tensor = torch.FloatTensor(X_normalized)
dataset = TensorDataset(X_tensor)
train_loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0)

print(f"📦 Data prepared for training: {len(dataset)} samples")

In [ ]:
# Train Deep Autoencoder
print("\n" + "="*50)
print("Training Deep Autoencoder")
print("="*50)

autoencoder = DeepAutoEncoder(
    input_dim=X_normalized.shape[1], 
    latent_dim=32, 
    hidden_dims=[512, 256, 128]
)

ae_losses = train_autoencoder(
    autoencoder, train_loader, epochs=50, lr=0.001, device=device
)

# Get embeddings from trained autoencoder
autoencoder.eval()
with torch.no_grad():
    ae_embeddings = []
    for batch in train_loader:
        batch_tensor = batch[0].to(device)
        _, z = autoencoder(batch_tensor)
        ae_embeddings.append(z.cpu().numpy())
    
ae_embeddings = np.vstack(ae_embeddings)
print(f"✅ Autoencoder embeddings shape: {ae_embeddings.shape}")

In [ ]:
# Train Variational Autoencoder
print("\n" + "="*50)
print("Training Variational Autoencoder")
print("="*50)

vae = VariationalAutoEncoder(
    input_dim=X_normalized.shape[1], 
    latent_dim=32, 
    hidden_dims=[512, 256]
)

vae_losses = train_autoencoder(
    vae, train_loader, epochs=50, lr=0.001, device=device
)

# Get embeddings from trained VAE
vae.eval()
with torch.no_grad():
    vae_embeddings = []
    for batch in train_loader:
        batch_tensor = batch[0].to(device)
        _, mu, logvar, z = vae(batch_tensor)
        vae_embeddings.append(mu.cpu().numpy())  # Use mean for deterministic embedding
    
vae_embeddings = np.vstack(vae_embeddings)
print(f"✅ VAE embeddings shape: {vae_embeddings.shape}")

In [ ]:
# Visualize training progress and embeddings
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=["AE Training Loss", "VAE Training Loss", "AE Embeddings (2D)", "VAE Embeddings (2D)"],
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# Training losses
fig.add_trace(
    go.Scatter(y=ae_losses, name="AE Loss", line=dict(color='blue')),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(y=vae_losses, name="VAE Loss", line=dict(color='red')),
    row=1, col=2
)

# 2D visualizations using PCA on embeddings
pca_2d = PCA(n_components=2, random_state=42)

# AE embeddings
ae_2d = pca_2d.fit_transform(ae_embeddings)
fig.add_trace(
    go.Scatter(
        x=ae_2d[:, 0], y=ae_2d[:, 1], 
        mode='markers', 
        marker=dict(color=y, colorscale='viridis', size=4),
        name="AE Embeddings",
        showlegend=False
    ),
    row=2, col=1
)

# VAE embeddings
vae_2d = pca_2d.fit_transform(vae_embeddings)
fig.add_trace(
    go.Scatter(
        x=vae_2d[:, 0], y=vae_2d[:, 1], 
        mode='markers', 
        marker=dict(color=y, colorscale='viridis', size=4),
        name="VAE Embeddings",
        showlegend=False
    ),
    row=2, col=2
)

fig.update_layout(
    title="Deep Learning Dimensionality Reduction Results",
    height=800,
    showlegend=True
)

fig.show()

print(f"🎨 Visualizations created for both autoencoder and VAE results")

<a id='5'></a>
## 5. Neural Network-Based Feature Extraction

Build and train deep neural networks to extract meaningful features for clustering.

In [ ]:
class FeatureExtractor(nn.Module):
    """Deep neural network for feature extraction"""
    
    def __init__(self, input_dim, output_dim=64, hidden_dims=[512, 256, 128]):
        super(FeatureExtractor, self).__init__()
        
        layers = []
        in_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(in_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3)
            ])
            in_dim = hidden_dim
        
        # Final feature layer
        layers.append(nn.Linear(in_dim, output_dim))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)


class ContrastiveLearning(nn.Module):
    """Contrastive learning for feature extraction"""
    
    def __init__(self, feature_extractor, temperature=0.07):
        super(ContrastiveLearning, self).__init__()
        self.feature_extractor = feature_extractor
        self.temperature = temperature
    
    def forward(self, x1, x2):
        z1 = F.normalize(self.feature_extractor(x1), dim=1)
        z2 = F.normalize(self.feature_extractor(x2), dim=1)
        return z1, z2
    
    def contrastive_loss(self, z1, z2):
        """NT-Xent loss for contrastive learning"""
        batch_size = z1.shape[0]
        
        # Normalize features
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)
        
        # Concatenate z1 and z2
        z = torch.cat([z1, z2], dim=0)
        
        # Compute similarity matrix
        sim_matrix = torch.mm(z, z.t()) / self.temperature
        
        # Remove diagonal elements
        sim_matrix = sim_matrix - torch.eye(2 * batch_size, device=z.device) * 1e9
        
        # Create positive pairs mask
        positive_pairs = torch.cat([
            torch.arange(batch_size, 2 * batch_size, device=z.device),
            torch.arange(0, batch_size, device=z.device)
        ])
        
        # Compute loss
        numerator = sim_matrix[torch.arange(2 * batch_size, device=z.device), positive_pairs]
        denominator = torch.logsumexp(sim_matrix, dim=1)
        
        loss = -numerator + denominator
        return loss.mean()


def create_augmented_data(X, noise_factor=0.1, dropout_rate=0.1):
    """Create augmented versions of the data for contrastive learning"""
    n_samples, n_features = X.shape
    
    # Augmentation 1: Add Gaussian noise
    X1 = X + np.random.normal(0, noise_factor, X.shape)
    
    # Augmentation 2: Feature dropout
    X2 = X.copy()
    dropout_mask = np.random.random(X.shape) < dropout_rate
    X2[dropout_mask] = 0
    
    return X1, X2


# Create feature extractor
feature_extractor = FeatureExtractor(
    input_dim=X_normalized.shape[1],
    output_dim=64,
    hidden_dims=[512, 256, 128]
)

# Setup contrastive learning
contrastive_model = ContrastiveLearning(feature_extractor)
contrastive_model.to(device)

print(f"🧠 Feature extractor created with output dimension: 64")
print(f"🔗 Contrastive learning model ready")

In [ ]:
# Train feature extractor with contrastive learning
def train_contrastive_learning(model, X, epochs=50, batch_size=128, lr=0.001):
    """Train feature extractor using contrastive learning"""
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    losses = []
    n_samples = X.shape[0]
    
    print(f"🚀 Training contrastive learning for {epochs} epochs")
    
    for epoch in range(epochs):
        epoch_loss = 0
        n_batches = 0
        
        for i in range(0, n_samples, batch_size):
            batch_end = min(i + batch_size, n_samples)
            batch_X = X[i:batch_end]
            
            # Create augmented data
            X1, X2 = create_augmented_data(batch_X)
            
            X1_tensor = torch.FloatTensor(X1).to(device)
            X2_tensor = torch.FloatTensor(X2).to(device)
            
            optimizer.zero_grad()
            
            z1, z2 = model(X1_tensor, X2_tensor)
            loss = model.contrastive_loss(z1, z2)
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        scheduler.step()
        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d}/{epochs}: Loss = {avg_loss:.6f}")
    
    print(f"✅ Contrastive learning training completed!")
    return losses


# Train the model
contrastive_losses = train_contrastive_learning(
    contrastive_model, X_normalized, epochs=30, batch_size=128, lr=0.001
)

# Extract features using trained model
contrastive_model.eval()
with torch.no_grad():
    X_tensor = torch.FloatTensor(X_normalized).to(device)
    contrastive_features = contrastive_model.feature_extractor(X_tensor).cpu().numpy()

print(f"✅ Contrastive features extracted: {contrastive_features.shape}")

In [ ]:
# Compare different dimensionality reduction techniques
from sklearn.manifold import TSNE
import umap

def apply_dimensionality_reduction(X, method='pca', n_components=2, **kwargs):
    """Apply various dimensionality reduction techniques"""
    if method == 'pca':
        reducer = PCA(n_components=n_components, random_state=42)
    elif method == 'tsne':
        reducer = TSNE(n_components=n_components, random_state=42, **kwargs)
    elif method == 'umap':
        reducer = umap.UMAP(n_components=n_components, random_state=42, **kwargs)
    else:
        raise ValueError(f"Unknown method: {method}")
    
    return reducer.fit_transform(X)

# Apply different reduction techniques
print("🔄 Applying various dimensionality reduction techniques...")

# Original data (high-dimensional)
original_pca = apply_dimensionality_reduction(X_normalized, 'pca')
original_tsne = apply_dimensionality_reduction(X_normalized[:1000], 'tsne')  # Subset for t-SNE speed
original_umap = apply_dimensionality_reduction(X_normalized, 'umap', n_neighbors=15, min_dist=0.1)

# Autoencoder embeddings
ae_pca = apply_dimensionality_reduction(ae_embeddings, 'pca')
ae_tsne = apply_dimensionality_reduction(ae_embeddings[:1000], 'tsne')
ae_umap = apply_dimensionality_reduction(ae_embeddings, 'umap', n_neighbors=15, min_dist=0.1)

# VAE embeddings
vae_pca = apply_dimensionality_reduction(vae_embeddings, 'pca')
vae_tsne = apply_dimensionality_reduction(vae_embeddings[:1000], 'tsne')
vae_umap = apply_dimensionality_reduction(vae_embeddings, 'umap', n_neighbors=15, min_dist=0.1)

# Contrastive features
contrast_pca = apply_dimensionality_reduction(contrastive_features, 'pca')
contrast_tsne = apply_dimensionality_reduction(contrastive_features[:1000], 'tsne')
contrast_umap = apply_dimensionality_reduction(contrastive_features, 'umap', n_neighbors=15, min_dist=0.1)

print("✅ All dimensionality reductions completed!")

In [ ]:
# Create comprehensive visualization comparing all methods
fig = make_subplots(
    rows=4, cols=3,
    subplot_titles=[
        "Original Data - PCA", "Original Data - t-SNE", "Original Data - UMAP",
        "Autoencoder - PCA", "Autoencoder - t-SNE", "Autoencoder - UMAP",
        "VAE - PCA", "VAE - t-SNE", "VAE - UMAP",
        "Contrastive - PCA", "Contrastive - t-SNE", "Contrastive - UMAP"
    ],
    vertical_spacing=0.08,
    horizontal_spacing=0.05
)

# Define color mapping for true labels
colors = y  # True labels for coloring
colors_subset = y[:1000]  # For t-SNE plots

# Row 1: Original Data
fig.add_trace(go.Scatter(x=original_pca[:, 0], y=original_pca[:, 1], 
                        mode='markers', marker=dict(color=colors, colorscale='viridis', size=3),
                        showlegend=False), row=1, col=1)

fig.add_trace(go.Scatter(x=original_tsne[:, 0], y=original_tsne[:, 1], 
                        mode='markers', marker=dict(color=colors_subset, colorscale='viridis', size=3),
                        showlegend=False), row=1, col=2)

fig.add_trace(go.Scatter(x=original_umap[:, 0], y=original_umap[:, 1], 
                        mode='markers', marker=dict(color=colors, colorscale='viridis', size=3),
                        showlegend=False), row=1, col=3)

# Row 2: Autoencoder
fig.add_trace(go.Scatter(x=ae_pca[:, 0], y=ae_pca[:, 1], 
                        mode='markers', marker=dict(color=colors, colorscale='viridis', size=3),
                        showlegend=False), row=2, col=1)

fig.add_trace(go.Scatter(x=ae_tsne[:, 0], y=ae_tsne[:, 1], 
                        mode='markers', marker=dict(color=colors_subset, colorscale='viridis', size=3),
                        showlegend=False), row=2, col=2)

fig.add_trace(go.Scatter(x=ae_umap[:, 0], y=ae_umap[:, 1], 
                        mode='markers', marker=dict(color=colors, colorscale='viridis', size=3),
                        showlegend=False), row=2, col=3)

# Row 3: VAE
fig.add_trace(go.Scatter(x=vae_pca[:, 0], y=vae_pca[:, 1], 
                        mode='markers', marker=dict(color=colors, colorscale='viridis', size=3),
                        showlegend=False), row=3, col=1)

fig.add_trace(go.Scatter(x=vae_tsne[:, 0], y=vae_tsne[:, 1], 
                        mode='markers', marker=dict(color=colors_subset, colorscale='viridis', size=3),
                        showlegend=False), row=3, col=2)

fig.add_trace(go.Scatter(x=vae_umap[:, 0], y=vae_umap[:, 1], 
                        mode='markers', marker=dict(color=colors, colorscale='viridis', size=3),
                        showlegend=False), row=3, col=3)

# Row 4: Contrastive Learning
fig.add_trace(go.Scatter(x=contrast_pca[:, 0], y=contrast_pca[:, 1], 
                        mode='markers', marker=dict(color=colors, colorscale='viridis', size=3),
                        showlegend=False), row=4, col=1)

fig.add_trace(go.Scatter(x=contrast_tsne[:, 0], y=contrast_tsne[:, 1], 
                        mode='markers', marker=dict(color=colors_subset, colorscale='viridis', size=3),
                        showlegend=False), row=4, col=2)

fig.add_trace(go.Scatter(x=contrast_umap[:, 0], y=contrast_umap[:, 1], 
                        mode='markers', marker=dict(color=colors, colorscale='viridis', size=3),
                        showlegend=False), row=4, col=3)

fig.update_layout(
    title="Comprehensive Comparison of Dimensionality Reduction Techniques",
    height=1200,
    showlegend=False
)

# Remove axis labels for cleaner look
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

fig.show()

print("🎨 Comprehensive visualization of all dimensionality reduction methods created!")